# 3 — Embeddings (Vertex AI)

**AWS-side equivalent:** Titan Embeddings V2 (1024 dim)
**GCP service:** Vertex AI `gemini-embedding-001` (default 3072d, configurable
down to 768 / 1536 via Matryoshka). Legacy `text-embedding-005` (768d) is
available for retrieval-quality A/B.

AWS-side notebook: `notebooks/aws/03-embeddings.ipynb`

**Dim mismatch with Titan:** OrgIQ's shared schema declares `vector(1024)`.
Notebook 02 already patched the column to `vector(<cfg.gcp.embedding_dim>)`
before applying. If you re-ran the schema by hand, run that step again or
the `INSERT ... CAST(... AS vector)` calls here will error on dim mismatch.

## Steps

1. Resolve embedding model + dim from `config.json`
2. Load people from AlloyDB (fall back to `data/org/people.json` if AlloyDB unreachable)
3. Batch-embed `name + title + team` summaries via the new `google-genai` client
4. Upsert into `node_embeddings`
5. Verify with a kNN query (`<=>` cosine distance)
6. Persist `gcp.embedding_dim` to `config.json` for downstream notebooks

## Step 1: Load config + open clients

In [ ]:
import json
from pathlib import Path

config_path = Path('config.json')
config = json.loads(config_path.read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'

EMBED_MODEL = gcp.get('embedding_model', 'gemini-embedding-001')
EMBED_DIM = int(gcp.get('embedding_dim', 1536))
TASK_TYPE_DOC = 'RETRIEVAL_DOCUMENT'
TASK_TYPE_QUERY = 'RETRIEVAL_QUERY'

print(f'project       : {PROJECT_ID}')
print(f'region        : {REGION}')
print(f'embed model   : {EMBED_MODEL}')
print(f'embed dim     : {EMBED_DIM}')

In [ ]:
from google import genai
from google.genai.types import EmbedContentConfig

client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

def embed_texts(texts: list[str], task_type: str = TASK_TYPE_DOC) -> list[list[float]]:
    """Embed a batch of texts and return one vector per input.

    gemini-embedding-001 accepts up to 250 inputs per call. We chunk
    defensively at 64 to stay well under quota.
    """
    out: list[list[float]] = []
    for i in range(0, len(texts), 64):
        chunk = texts[i:i + 64]
        resp = client.models.embed_content(
            model=EMBED_MODEL,
            contents=chunk,
            config=EmbedContentConfig(
                task_type=task_type,
                output_dimensionality=EMBED_DIM,
            ),
        )
        out.extend(e.values for e in resp.embeddings)
    return out

# Smoke test.
_v = embed_texts(['smoke test'])[0]
assert len(_v) == EMBED_DIM, f'Expected dim {EMBED_DIM}, got {len(_v)}'
print(f'✓ embed_texts ready (dim={len(_v)})')

## Step 2: Open AlloyDB connection (with JSON fallback)

If AlloyDB isn't provisioned yet (notebook 02 wasn't run), this falls back
to reading `data/org/people.json` so the embedding step can still be
exercised. The fallback path writes vectors to a local Parquet file instead
of the database so we don't lose work.

In [ ]:
from contextlib import contextmanager

INSTANCE_URI = gcp.get('alloydb_instance_uri')
DB_NAME = gcp.get('alloydb_db', 'orgiq')
DB_USER = gcp.get('alloydb_user', 'postgres')
PWD_SECRET = gcp.get('alloydb_password_secret', 'orgiq-alloydb-password')

_have_alloydb = False
_connector = None
_db_password = None

if INSTANCE_URI:
    try:
        from google.cloud import secretmanager
        from google.cloud.alloydbconnector import Connector
        sm = secretmanager.SecretManagerServiceClient()
        _db_password = sm.access_secret_version(
            request={'name': f'projects/{PROJECT_ID}/secrets/{PWD_SECRET}/versions/latest'}
        ).payload.data.decode()
        _connector = Connector()
        _have_alloydb = True
        print(f'✓ AlloyDB connector ready ({INSTANCE_URI})')
    except Exception as e:
        print(f'⚠ AlloyDB not reachable ({type(e).__name__}: {e}). Falling back to JSON.')
else:
    print('⚠ gcp.alloydb_instance_uri not set — using JSON fallback')

@contextmanager
def alloydb_conn():
    import psycopg
    if not _have_alloydb:
        raise RuntimeError('AlloyDB not configured')
    conn = _connector.connect(
        INSTANCE_URI, 'psycopg',
        user=DB_USER, password=_db_password, db=DB_NAME, ip_type='PUBLIC',
    )
    try:
        yield conn
    finally:
        conn.close()

## Step 3: Load people rows

In [ ]:
if _have_alloydb:
    with alloydb_conn() as conn, conn.cursor() as cur:
        cur.execute('SELECT id, name, title, team FROM people ORDER BY id')
        person_rows = [
            {'id': r[0], 'name': r[1], 'title': r[2], 'team': r[3]}
            for r in cur.fetchall()
        ]
    print(f'✓ Loaded {len(person_rows)} people from AlloyDB')
else:
    raw = json.loads(Path('data/org/people.json').read_text())
    person_rows = [
        {
            'id': p['id'],
            'name': p['name'],
            'title': p.get('title', ''),
            'team': p.get('department') or p.get('team', ''),
        }
        for p in raw
    ]
    print(f'✓ Loaded {len(person_rows)} people from data/org/people.json (fallback)')

print(f'  sample: {person_rows[0]}')

## Step 4: Build summary strings + batch embed

In [ ]:
def profile(p: dict) -> str:
    title = p.get('title') or 'Unknown role'
    team = p.get('team') or 'Unknown team'
    return f"{p['name']}, {title}. Team: {team}."

texts = [profile(p) for p in person_rows]
print(f'⏳ Embedding {len(texts)} person summaries…')
vectors = embed_texts(texts, task_type=TASK_TYPE_DOC)
print(f'✓ Got {len(vectors)} vectors, each dim {len(vectors[0])}')

## Step 5: Upsert into `node_embeddings` (or local Parquet on fallback)

In [ ]:
def _vec_literal(v: list[float]) -> str:
    return '[' + ','.join(f'{x:.8f}' for x in v) + ']'

if _have_alloydb:
    with alloydb_conn() as conn:
        conn.autocommit = False
        with conn.cursor() as cur:
            for p, text, vec in zip(person_rows, texts, vectors):
                cur.execute(
                    """
                    INSERT INTO node_embeddings (node_id, node_type, content, embedding)
                    VALUES (%(id)s, 'Person', %(content)s, %(emb)s::vector)
                    ON CONFLICT (node_id) DO UPDATE
                      SET node_type = EXCLUDED.node_type,
                          content = EXCLUDED.content,
                          embedding = EXCLUDED.embedding
                    """,
                    {'id': p['id'], 'content': text, 'emb': _vec_literal(vec)},
                )
        conn.commit()
    print(f'✓ Upserted {len(vectors)} rows into node_embeddings')
else:
    import pandas as pd
    out_path = Path('data/embeddings_people.parquet')
    out_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({
        'node_id': [p['id'] for p in person_rows],
        'node_type': 'Person',
        'content': texts,
        'embedding': vectors,
    }).to_parquet(out_path)
    print(f'⚠ AlloyDB not available; wrote {len(vectors)} rows to {out_path}')

## Step 6: kNN sanity check

Pick a query, embed with `RETRIEVAL_QUERY` task type (gemini-embedding-001
uses different prompts for queries vs documents), and find the top-3
nearest people.

In [ ]:
query_text = 'Who knows the most about container orchestration and platform reliability?'
q_vec = embed_texts([query_text], task_type=TASK_TYPE_QUERY)[0]

if _have_alloydb:
    with alloydb_conn() as conn, conn.cursor() as cur:
        cur.execute(
            """
            SELECT node_id, content, embedding <=> %s::vector AS cosine_dist
            FROM   node_embeddings
            WHERE  node_type = 'Person'
            ORDER  BY cosine_dist
            LIMIT  3
            """,
            (_vec_literal(q_vec),),
        )
        rows = cur.fetchall()
    print(f'Query: {query_text!r}')
    for node_id, content, dist in rows:
        print(f'  [{node_id}]  cosine={dist:.4f}  {content}')
else:
    # Local fallback: numpy cosine on the in-memory vectors.
    import numpy as np
    arr = np.array(vectors)
    q = np.array(q_vec)
    dists = 1 - (arr @ q) / (np.linalg.norm(arr, axis=1) * np.linalg.norm(q))
    top = np.argsort(dists)[:3]
    print(f'Query: {query_text!r} (local numpy fallback)')
    for i in top:
        print(f'  [{person_rows[i]["id"]}]  cosine={dists[i]:.4f}  {texts[i]}')

## Step 7: Persist embedding config

`embedding_dim` was set by notebook 01 and consumed by notebook 02 when
applying the schema. We re-write it here defensively so a re-run of
notebook 03 with a different dim updates `config.json` cleanly.

In [ ]:
gcp['embedding_model'] = EMBED_MODEL
gcp['embedding_dim'] = EMBED_DIM
gcp['embeddings_table'] = 'node_embeddings'
config_path.write_text(json.dumps(config, indent=2))
print('✓ config.json updated:')
print(f'  gcp.embedding_model = {EMBED_MODEL}')
print(f'  gcp.embedding_dim   = {EMBED_DIM}')
print(f'  gcp.embeddings_table = node_embeddings')

if _connector is not None:
    _connector.close()

## Summary

- [x] `google-genai` client wired to Vertex AI (project + region scoped)
- [x] `gemini-embedding-001` @ `gcp.embedding_dim` dimensions
- [x] Distinct task types for documents (`RETRIEVAL_DOCUMENT`) and queries (`RETRIEVAL_QUERY`)
- [x] Upsert into `node_embeddings` via psycopg through the AlloyDB connector
- [x] Parquet fallback when AlloyDB isn't provisioned
- [x] kNN sanity query using pgvector's `<=>` cosine operator

**Next:** `04-vertex-search.ipynb` — Discovery Engine data store + search app.